In [1]:
# Making a list of equivalent names for medicines.

In [2]:
import pandas as pd
import re, json
from collections import defaultdict

# ---------------------------------------------------------------------------
# 1. Normalise text in a single, repeatable way
# ---------------------------------------------------------------------------
_CLEAN_RX = re.compile(r'"[^"]*"')

def clean(text: str) -> str:
    """Remove text in double quotes, collapse whitespace, lowercase."""
    text = _CLEAN_RX.sub('', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# ---------------------------------------------------------------------------
# 2. Read only the required columns
# ---------------------------------------------------------------------------
discont = pd.read_excel(
    'Afregistrerede_Laegemidler.xls',
    sheet_name='AfregLægemidlerSidsteÅr',
    usecols=['Lægemiddel', 'AktiveSubstanser']
)

current = pd.read_excel(
    'ListeOverGodkendteLaegemidler.xlsx',
    sheet_name='Godkendte Lægemidler',
    usecols=['Navn', 'AktiveSubstanser']
)

# ---------------------------------------------------------------------------
# 3. Build: generic_key ➜ set{all equivalent names}
# ---------------------------------------------------------------------------
equiv: defaultdict[str, set[str]] = defaultdict(set)

def add_row(syn_raw, gen_raw):
    if pd.isna(syn_raw) or pd.isna(gen_raw):
        return
    syn = clean(syn_raw)
    gen = clean(gen_raw)
    if not syn or not gen:
        return
    equiv[gen].update([gen, syn])   # always include the generic itself

for syn, gen in zip(discont['Lægemiddel'], discont['AktiveSubstanser']):
    add_row(syn, gen)

for syn, gen in zip(current['Navn'], current['AktiveSubstanser']):
    add_row(syn, gen)

# ---------------------------------------------------------------------------
# 4. Convert to list-of-lists (each list sorted for readability)
# ---------------------------------------------------------------------------
equivalence_groups = [sorted(list(names)) for names in equiv.values()]

# Optional: sort the groups themselves (shortest first then lexicographically)
equivalence_groups.sort(key=lambda lst: (len(lst[0]), lst))

# ---------------------------------------------------------------------------
# 5. Write to JSON
# ---------------------------------------------------------------------------
with open('equivalent_names.json', 'w', encoding='utf-8') as fh:
    json.dump(equivalence_groups, fh, ensure_ascii=False, indent=2)

print(f"Created {len(equivalence_groups)} groups in equivalent_names.json")

c:\Users\danam\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Created 2910 groups in equivalent_names.json
